<a href="https://colab.research.google.com/github/jjmoncus/MV_coursework_2/blob/main/Machine_Vision_Final_Lab_Model_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Confirm using python 3.12

In [1]:
import sys
print(sys.version)

3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:07:49) [Clang 20.1.8 ]


In [2]:
# !pip install opencv-python
# !pip install boto3 -q
# !pip install torch
# !pip install numpy 
# !pip install torchvision
# !pip install seaborn
# !pip install huggingface_hub
# !pip install mediapipe==0.10.13

# Unified installation for the 2026 AI Stack
# !pip install  \
#     torch==2.9.1 \
#     torchvision==0.24.1 \
#     numpy==2.2.1 \
#     mediapipe==0.10.30 \
#     opencv-python==4.12.0.88 \
#     boto3==1.35.0 \
#     huggingface_hub \
#     seaborn \
#     tqdm

!pip install \
    opencv-python==4.12.0.88 \
    numpy==2.0.2 \
    torch==2.9.0 \
    seaborn==0.13.2 \
    pandas==2.2.2 \
    matplotlib==3.10.0 \
    huggingface_hub==0.36.0 \
    tqdm==4.67.1 \
    boto3==1.42.24 \
    mediapipe==0.10.30

# Load imports

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import ReduceLROnPlateau

from huggingface_hub import hf_hub_download

import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os


import copy
import cv2
import numpy as np
import shutil
import random
import psutil

from tqdm import tqdm
import time
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

import mediapipe as mp

import warnings

import subprocess, sys
target_version = "0.10.30"
if mp.__version__ == target_version:
    print(f"✅ MediaPipe {target_version} is already installed and ready!")
else:
    print(f"⚠️ Version mismatch (Found {mp.__version__}). Attempting install...")
    # Using sys.executable ensures pip installs to the CORRECT python version
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"mediapipe=={target_version}"])

# Suppress Protobuf and MediaPipe internal warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 
warnings.filterwarnings("ignore", category=UserWarning, module='google.protobuf.symbol_database')

✅ MediaPipe 0.10.30 is already installed and ready!


# Show what versions of everything I'm using

In [4]:
print(f"Python: {sys.version}")
print(f"torch: {torch.__version__}")
print(f"boto3: {boto3.__version__}")
print(f"cv2: {cv2.__version__}")
print(f"numpy: {np.__version__}")
print(f"mediapipe: {mp.__version__}")

Python: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:07:49) [Clang 20.1.8 ]
torch: 2.9.1
boto3: 1.42.19
cv2: 4.12.0
numpy: 2.2.6
mediapipe: 0.10.30


# Find and assign device

In [5]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


# Please double, triple, quadruple check that the below code runs without errors before submitting.

## TODO 1 - Enter your HuggingFace username below:

In [6]:
hf_username = "jjmoncus1"

## TODO 2 - Define your model EXACTLY as you did in your training code (otherwise there will be errors, and, possibly, tears).

Note below the classname is 'YourModelArchitecture'. That's because it literally needs to be YOUR MODEL ARCHITECTURE. This class definition is later referred to below in the 'load_model_from_hub' method. The architecture must match here, or it will not be able to instantiate the model weights correctly once it downloads them from HuggingFace. Pay very close attention to getting this right, please.

Replace the below code, and replace the corresponding line in the 'load_model_from_hub' method.

In [7]:
class PoseModel(nn.Module):
    def __init__(self, num_joints=33, num_features=4, num_classes=10):
        super().__init__()
        
        # Total input channels = joints * features (e.g., 33 * 4 = 132)
        self.in_channels = num_joints * num_features


        self.input_bn = nn.BatchNorm1d(self.in_channels)

        # initial FC for compressing the joint/feature information
        self.fc_1 = nn.Sequential(

            nn.Linear(self.in_channels, 64),
            nn.LeakyReLU(0.1)
        )


        # process the joint/feature info (channels) over time
        self.temporal_conv = nn.Sequential(
            
            # Layer 1: [B, 132, 100] ---> [B, 64, 50]
            nn.Conv1d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.1),

            # Layer 2: [B, 64, 50] ---> [B, 128, 25]
            nn.Conv1d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.1),

            # Layer 3: [B, 128, 25] ---> [B, 256, 13]
            nn.Conv1d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.1),

            # Global Average Pooling: [B, 256, 13] ---> [B, 256, 1]
            nn.AdaptiveAvgPool1d(1)
        )

        self.avg_pool = nn.AdaptiveAvgPool1d(1)  # [B, 256, 13] ---> [B, 256, 1]
        self.max_pool = nn.AdaptiveMaxPool1d(1)  # [B, 256, 13] ---> [B, 256, 1]
        
        # Output of temporal is [B, 256, 2]. Flattened = 512
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.1),
            nn.Linear(256, 112),
            nn.LeakyReLU(0.1),
            nn.Dropout(p=0.2),
            nn.Linear(112, num_classes),
        )

    def forward(self, x):
        """
        Input x: [Batch, Time, Joints, Features]
        """
        
        B, T, J, F = x.shape

        # flatten joints and features into channel dimension
        x = x.view(B, T, J * F)       # [B, T, 132]

        # batch norm before first linear layer
        x = x.transpose(1, 2)          # [B, 132, T]
        x = self.input_bn(x)           # Normalize across the batch/time
        x = x.transpose(1, 2)          # Back to [B, T, 132]

        # send through FC_1
        x = self.fc_1(x)              # [B, T, 32]

        # move dimension
        x = x.transpose(1, 2)         # [B, 32, T]
        
        # send through temporal layers
        x = self.temporal_conv(x)     # [B, 256, 13]

        x_avg = self.avg_pool(x) # [B, 256, 13]
        x_max = self.max_pool(x) # [B, 256, 13]

        x = torch.cat([x_avg, x_max], dim = 2) # [B, 256, 2]
        
        # classify
        return self.classifier(x)     # [B, 10, 1]


## Download the test data from s3, and create the corresponding dataset + dataloader.

There's no TODO for you here. This text is just here to explain to you what this code does.

In this instance, the test data IS the training data you were provided in the Model Training notebook. This is by design. You do not have access to the test data. This is a simple check to make sure the mechanics of this notebook work.

You should achieve the same accuracy here in this notebook, as you did in your previous notebook (random seed notwithstanding).

In [8]:
# =============================================================================
# DOWNLOAD TEST DATA FROM S3
# =============================================================================

def download_test_data(bucket_name='training-and-validation-data',download_dir='./test-data'):
    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

    bucket_name = 'prism-mvta'
    prefix = 'training-and-validation-data/'

    os.makedirs(download_dir, exist_ok=True)

    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    video_names = []

    for page in pages:
        if 'Contents' not in page:
            print("No files found at the specified path!")
            break

        print("Downloading test data:\n")
        for obj in tqdm(page['Contents']):
            key = obj['Key']
            filename = os.path.basename(key)

            if not filename:
                continue

            video_names.append(filename)
            local_path = os.path.join(download_dir, filename)
            # print(f"Downloading: {filename}")
            s3.download_file(bucket_name, key, local_path)

    print(f"\nDownloaded {len(video_names)} test videos")
    return download_dir
    

# Run Test Data Through Pose Estimation

Function for extracting mediapope joint coordinates

Function for batch extracting poses from a bunch of videos

In [ ]:
# for a given video, return the z-positions of their right shoulder
from scipy.signal import savgol_filter

def smooth_landmarks_savgol(tensor, window_size=7, poly_order=2):
    """
    Input: tensor of shape (T, 33, 4)
    window_size: Must be odd. Larger = smoother but may lag.
    poly_order: Typically 2 or 3.
    """
    # Convert to numpy for scipy
    data = tensor.clone().cpu().numpy()
    
    # We iterate over joints (33) and features (4: x, y, z, vis)
    for j in range(33):
        for f in range(4):
            # Apply filter along the Time (T) dimension
            data[:, j, f] = savgol_filter(data[:, j, f], window_size, poly_order)
            
    return torch.from_numpy(data)


# def flip_pose_tensor(tensor):
#     """
#     Input: tensor of shape (T, 33, 4) - [Time, Joints, (x, y, z, vis)]
#     Returns: flipped_tensor of same shape
#     """
    
#     flipped_data = tensor.clone()

#     # Invert the x-axis
#     flipped_data[:, :, 0] = -flipped_data[:, :, 0]

#     # Define left/right pairs to swap
#     pairs = [
#         (1, 4), (2, 5), (3, 6),    # Eyes
#         (7, 8),                    # Ears
#         (9, 10),                   # Mouth
#         (11, 12),                  # Shoulders
#         (13, 14),                  # Elbows
#         (15, 16),                  # Wrists
#         (17, 18), (19, 20), (21, 22), # Hands/Fingers
#         (23, 24),                  # Hips
#         (25, 26),                  # Knees
#         (27, 28),                  # Ankles
#         (29, 30),                  # Heels
#         (31, 32)                   # Toes
#     ]

#     # make swaps
#     for left_idx, right_idx in pairs:
        
#         # temporary storage of left side
#         temp_left = flipped_data[:, left_idx, :].clone()
        
#         # move right to left
#         flipped_data[:, left_idx, :] = flipped_data[:, right_idx, :]
        
#         # move temp_left to right
#         flipped_data[:, right_idx, :] = temp_left

#     return flipped_data

# def export_poses(video_paths, search_dir, output_dir, smooth_window_size = 7):
#     """
#     Iterates through video_paths, extracts pose data, and saves as .pt tensors.
#     """
#     print("\nEstimating Poses in Training Videos...\n")
    
#     # Refresh output directory if exists
#     if os.path.exists(output_dir):
#         shutil.rmtree(output_dir)
#         print(f"{output_dir} already exists - refreshing now.")
#     else:
#         print(f"{output_dir} not found - creating.")
#     os.makedirs(output_dir, exist_ok=True)
    
#     # Initialize MediaPipe once
#     # mp_pose = mp.solutions.pose
#     # pose_processor = mp_pose.Pose(static_image_mode=False, min_detection_confidence=0.5)
#     pose_processor = vision.PoseLandmarker.create_from_options(options)

#     # Wrap the loop with tqdm for a progress bar
#     for video_name in tqdm(video_paths, desc="Processing Videos", unit="video"):
        
#         # Construct full input path
#         video_input_path = os.path.join(search_dir, video_name)
        
#         if not os.path.exists(video_input_path):
#             # Using tqdm.write prevents the progress bar from breaking
#             tqdm.write(f"⚠️ Skip: {video_name} not found in {search_dir}")
#             continue

#         # Extract landmarks using OpenCV loop
#         cap = cv2.VideoCapture(video_input_path)
#         all_landmarks = []
        
#         while cap.isOpened():
#             ret, frame = cap.read()
#             if not ret:
#                 break
            
#             frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#             results = pose_processor.process(frame_rgb)
            
#             current_frame = np.zeros((33, 4))
#             if results.pose_world_landmarks:
#                 for i, lm in enumerate(results.pose_world_landmarks.landmark):
#                     current_frame[i] = [lm.x, lm.y, lm.z, lm.visibility]
            
#             all_landmarks.append(current_frame)
        
#         cap.release()

#         # Convert to Torch Tensor [T, 33, 4]
#         video_tensor = torch.from_numpy(np.array(all_landmarks)).float()

#         # smooth using the right smoothing kernel
#         video_tensor = smooth_landmarks_savgol(video_tensor.clone(), window_size = smooth_window_size, poly_order = 2)

#         # save to output
#         base_name = os.path.splitext(video_name)[0]
#         output_filename = f"{base_name}_pose.pt"
#         save_path = os.path.join(output_dir, output_filename)
        
#         torch.save(video_tensor, save_path)

        
#         # immediately export the horizontally flipped version, too
#         flipped_video_tensor = flip_pose_tensor(video_tensor)
#         output_filename = f"{base_name}_FLIPPED_pose.pt"
#         save_path = os.path.join(output_dir, output_filename)
#         torch.save(flipped_video_tensor, save_path)
        

# # --- Constants for the new API ---
# MODEL_PATH = 'pose_landmarker_heavy.task' # Ensure this file is in your directory

# def export_poses(video_paths, search_dir, output_dir, smooth_window_size=7):
#     """
#     Iterates through video_paths, extracts pose data using Tasks API, and saves as .pt tensors.
#     """
#     print("\nEstimating Poses using MediaPipe Tasks API...\n")
    
#     # Refresh output directory if exists
#     if os.path.exists(output_dir):
#         shutil.rmtree(output_dir)
#         print(f"{output_dir} already exists - refreshing now.")
#     else:
#         print(f"{output_dir} not found - creating.")
#     os.makedirs(output_dir, exist_ok=True)

#     # 1. Initialize the MediaPipe Pose Landmarker for VIDEO mode
#     BaseOptions = mp.tasks.BaseOptions
#     PoseLandmarker = mp.tasks.vision.PoseLandmarker
#     PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
#     VisionRunningMode = mp.tasks.vision.RunningMode

#     options = PoseLandmarkerOptions(
#         base_options=BaseOptions(model_asset_path=MODEL_PATH),
#         running_mode=VisionRunningMode.VIDEO,
#         num_poses=1,
#         min_pose_detection_confidence=0.5,
#         min_pose_presence_confidence=0.5,
#         min_tracking_confidence=0.5
#     )

#     # For every video ...
#     for video_name in tqdm(video_paths, desc="Processing Videos", unit="video"):
        
#         # !!! CRITICAL: Create a NEW landmarker for EVERY video !!!
#         # This resets the internal timestamp counter to 0 for each new file.
#         with PoseLandmarker.create_from_options(options) as landmarker:
#             video_input_path = os.path.join(search_dir, video_name)
            
#             if not os.path.exists(video_input_path):
#                 continue

#             cap = cv2.VideoCapture(video_input_path)
#             all_landmarks = []
#             last_timestamp_ms = -1 
            
#             while cap.isOpened():
#                 ret, frame = cap.read()
#                 if not ret:
#                     break
                
#                 # Using frame index * (1000/FPS) is often more stable than PROP_POS_MSEC
#                 # for certain video containers that report 0 for all frames.
#                 fps = cap.get(cv2.CAP_PROP_FPS)
#                 frame_idx = cap.get(cv2.CAP_PROP_POS_FRAMES)
#                 current_timestamp_ms = int((frame_idx * 1000) / fps)
                
#                 if current_timestamp_ms <= last_timestamp_ms:
#                     current_timestamp_ms = last_timestamp_ms + 1
                
#                 last_timestamp_ms = current_timestamp_ms

#                 frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#                 mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                
#                 results = landmarker.detect_for_video(mp_image, current_timestamp_ms)
                
#                 current_frame = np.zeros((33, 4))
                
#                 # Note: results.pose_world_landmarks is now a list of lists
#                 if results.pose_world_landmarks:
#                     # We take the first detected pose [0]
#                     for i, lm in enumerate(results.pose_world_landmarks[0]):
#                         current_frame[i] = [lm.x, lm.y, lm.z, lm.visibility]
                
#                 all_landmarks.append(current_frame)
#                 frame_idx += 1
            
#             cap.release()

#             if not all_landmarks:
#                 continue

#             # Convert to Tensor and process
#             video_tensor = torch.from_numpy(np.array(all_landmarks)).float()
#             video_tensor = smooth_landmarks_savgol(video_tensor, window_size=smooth_window_size)

#             # Save Original
#             base_name = os.path.splitext(video_name)[0]
#             torch.save(video_tensor, os.path.join(output_dir, f"{base_name}_pose.pt"))

#             # Save Flipped
#             flipped_video_tensor = flip_pose_tensor(video_tensor)
#             torch.save(flipped_video_tensor, os.path.join(output_dir, f"{base_name}_FLIPPED_pose.pt"))


from mediapipe.tasks import python
from mediapipe.tasks.python import vision

MODEL_PATH = 'pose_landmarker_lite.task'

def export_poses(video_paths, search_dir, output_dir, smooth_window_size=7):
    """
    Extracts pose data using Tasks API in IMAGE mode (frame-by-frame).
    This ensures 100% independence between videos and frames.
    """
    print("\nEstimating Poses using MediaPipe Tasks API (IMAGE Mode)...\n")
    
    # Refresh output directory if exists
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
        print(f"{output_dir} already exists - refreshing now.")
    else:
        print(f"{output_dir} not found - creating.")
    os.makedirs(output_dir, exist_ok=True)

    # 1. Setup Options for IMAGE mode
    BaseOptions = mp.tasks.BaseOptions
    PoseLandmarker = mp.tasks.vision.PoseLandmarker
    PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=VisionRunningMode.IMAGE, # <--- Changed to IMAGE
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5
        # min_tracking_confidence is ignored in IMAGE mode
    )

    # 2. Instantiate ONCE for all videos
    with PoseLandmarker.create_from_options(options) as landmarker:
        
        for video_name in tqdm(video_paths, desc="Processing Videos", unit="video"):
            video_input_path = os.path.join(search_dir, video_name)
            
            if not os.path.exists(video_input_path):
                continue

            cap = cv2.VideoCapture(video_input_path)
            all_landmarks = []
            
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                
                # Convert to MediaPipe Image
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
                
                # 3. Use .detect() - No timestamp required!
                results = landmarker.detect(mp_image)
                
                current_frame = np.zeros((33, 4))
                if results.pose_world_landmarks:
                    # Take the first detected pose
                    for i, lm in enumerate(results.pose_world_landmarks[0]):
                        current_frame[i] = [lm.x, lm.y, lm.z, lm.visibility]
                
                all_landmarks.append(current_frame)
            
            cap.release()

            if not all_landmarks:
                continue

            # --- Post-Processing (Same as before) ---
            video_tensor = torch.from_numpy(np.array(all_landmarks)).float()
            
            # Since IMAGE mode has slightly more jitter than VIDEO mode,
            # your Savgol filter is now even more important.
            video_tensor = smooth_landmarks_savgol(video_tensor, window_size=smooth_window_size)

            base_name = os.path.splitext(video_name)[0]
            torch.save(video_tensor, os.path.join(output_dir, f"{base_name}_pose.pt"))

            # flipped_video_tensor = flip_pose_tensor(video_tensor)
            # torch.save(flipped_video_tensor, os.path.join(output_dir, f"{base_name}_FLIPPED_pose.pt"))

# Data Loaders

In [10]:

class PoseDataset(Dataset):
    """Dataset for loading poses from a folder. Labels from filename prefix."""

    def __init__(self, pose_dir):
        self.pose_dir = pose_dir
        
        self.video_files = [ f for f in os.listdir(pose_dir)if f.endswith(('.pt')) ]
        self.labels = [ int(f.split('_')[0]) for f in self.video_files]

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_path = os.path.join(self.pose_dir, self.video_files[idx])
        frames = self._load_video(video_path)
        label = self.labels[idx]

        return frames, label

    def _load_video(self, path, stride = 1, max_frames = 1000):
        
        video = torch.load(path)
        frames = []
        frame_count = 0
        T = video.shape[0]

        for t in range(T):
            if t % stride == 0:
                frame = video[t, :, :]
                frames.append(frame)
            if len(frames) == max_frames:
                break

        frames = torch.from_numpy(np.array(frames))

        return frames


def collate_fn(batch):
    frames_list, labels = zip(*batch)
    target_frames = 1000

    padded_frames = []
    
    # For each video, ...
    for frames in frames_list:
        
        # frames shape: (T, 33, 4)
        num_frames = frames.shape[0]
        
        # if the video is too short, ...
        if num_frames < target_frames:

            # ... grab the last frame: (C, 1, H, W)
            last_frame = frames[-1:, :, :]
            
            # ... calculate how many times to repeat it
            padding_size = target_frames - num_frames
            
            # ... create the padding by repeating the last frame
            padding = last_frame.repeat(padding_size, 1, 1)
            
            # ..., and concatenate along the T dimension (0)
            frames = torch.cat([frames, padding], dim=0)
        
        # if video is too long ...
        elif num_frames > target_frames:
            
            # ... truncate to 100 frames
            frames = frames[:target_frames, :, :]

        # Finally, add our padded video to the list of videos
        padded_frames.append(frames)

    # Combine list into a batch tensor: (Batch, T, 33, 4)
    frames_batch = torch.stack(padded_frames, dim=0)
    labels_batch = torch.tensor(labels)

    return frames_batch, labels_batch



## TODO 3 - Download your model from HuggingFace and instantiate it

Replace line 8 of the below code. Line 8 is where you instantiate YOUR MODEL ARCHITECTURE (which you re-defined above) with the weights you download from HuggingFace. Make sure you get the class name, and the arguments to the __init__ method correct.


This code just downloads the same model which you uploaded in the last notebook.

In [11]:
# =============================================================================
# DOWNLOAD MODEL FROM HUGGING FACE
# =============================================================================

def load_model_from_hub(repo_id):
    model_path = hf_hub_download(repo_id=repo_id, filename="model.pt")

    model = PoseModel()
    model.load_state_dict(torch.load(model_path, map_location=device))

    print(f"Model loaded from {repo_id}")
    return model

model = load_model_from_hub(f"{hf_username}/mv-final-assignment")
model.float()

Model loaded from jjmoncus1/mv-final-assignment


PoseModel(
  (input_bn): BatchNorm1d(132, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc_1): Sequential(
    (0): Linear(in_features=132, out_features=64, bias=True)
    (1): LeakyReLU(negative_slope=0.1)
  )
  (temporal_conv): Sequential(
    (0): Conv1d(64, 64, kernel_size=(3,), stride=(2,), padding=(1,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.1)
    (3): Dropout(p=0.1, inplace=False)
    (4): Conv1d(64, 128, kernel_size=(3,), stride=(2,), padding=(1,))
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): LeakyReLU(negative_slope=0.1)
    (7): Dropout(p=0.1, inplace=False)
    (8): Conv1d(128, 256, kernel_size=(3,), stride=(2,), padding=(1,))
    (9): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): LeakyReLU(negative_slope=0.1)
    (11): AdaptiveAvgPool1d(output_size=1)
  )
  (avg_pool): A

## TODO 4

Make sure the below code correctly evaluates your model performance on the given data!

This is your last chance to verify this before submission.

In [12]:
def evaluate(model, test_loader, dataset, device):
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []
    all_times = []

    print("\n")

    with torch.no_grad():
        for idx, (frames, labels) in enumerate(test_loader):
            frames, labels = frames.to(device), labels.to(device)
            frames = frames.float() # MAYBE COME BACK TO THIS
            # labels = labels.float()

            # Time the forward pass
            start_time = time.time()
            outputs = model(frames)
            if device.type == 'cuda':
                torch.cuda.synchronize()  # wait for GPU to finish
            end_time = time.time()

            inference_time = (end_time - start_time) * 1000  # ms
            all_times.append(inference_time)

            _, preds = torch.max(outputs, 1)
            preds = preds + 1

            for i in range(labels.size(0)):
                batch_idx = idx * test_loader.batch_size + i
                video_name = dataset.video_files[batch_idx]
                pred = preds[i].item()
                true_label = labels[i].item()
                is_correct = "✓" if pred == true_label else "✗"

                print(f"{is_correct}  pred={pred}  true={true_label}  |  {inference_time:>7.1f}ms  |  {video_name}")

            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / total
    return accuracy, all_preds, all_labels, all_times


# =============================================================================
# RUN INFERENCE
# =============================================================================

def run_inference(model, bucket_name='training-and-validation-data'):

    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device("cpu")

    print(f"Using device: {device}")

    
    # Download test data
    test_dir = download_test_data(bucket_name, './test-data')

    # perform pose estimation on test-data, output to test-pose-data
    video_paths = [f for f in os.listdir("./test-data") if f.endswith(('.mp4', '.avi', '.mov'))]
    export_poses(video_paths, test_dir, "test-pose-data")

    model = model.to(device)

    # Create dataloader
    test_dataset = PoseDataset("test-pose-data")
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=collate_fn
    )

    print(f"\nRunning inference on {len(test_dataset)} test videos...")

    # Warmup (optional, helps get consistent GPU timings)
    if device.type == 'cuda':
        dummy = torch.randn(1, 1000, 33, 4).to(device)
        with torch.no_grad():
            _ = model(dummy)
        torch.cuda.synchronize()

    total_start = time.time()
    accuracy, preds, labels, times = evaluate(model, test_loader, test_dataset, device)
    total_end = time.time()

    # Summary
    num_correct = sum(p == l for p, l in zip(preds, labels))
    num_wrong = len(preds) - num_correct

    print("\n" + "="*50)
    print("SUMMARY")
    print("="*50)
    print(f"Total videos:         {len(preds)}")
    print(f"Correct:              {num_correct}")
    print(f"Incorrect:                {num_wrong}")
    print(f"")
    print(f"ACCURACY:             {accuracy*100:.2f}%")
    print(f"")
    print(f"Total time:           {total_end - total_start:.2f}s")
    print(f"Avg per video:        {sum(times) / len(times):.1f}ms")
    print(f"Min latency:          {min(times):.1f}ms")
    print(f"Max latency:          {max(times):.1f}ms")
    print("="*50)
    return accuracy, preds, labels

# setting a manual seed for reproducibility
torch.manual_seed(0)
_, _, _ = run_inference(model)

Using device: mps



100%|██████████| 77/77 [01:54<00:00,  1.49s/it]



Downloaded 77 test videos

Estimating Poses using MediaPipe Tasks API (IMAGE Mode)...

test-pose-data not found - creating.


I0000 00:00:1767843515.082439 8372848 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 88), renderer: Apple M1 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1767843515.196273 8372851 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1767843515.210228 8372849 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Processing Videos: 100%|██████████| 77/77 [06:10<00:00,  4.81s/video]



Running inference on 154 test videos...


✓  pred=3  true=3  |    715.8ms  |  3_kling_20251206_Text_to_Video_Generate_a_315_2_pose.pt
✓  pred=4  true=4  |      1.8ms  |  4_20251209_Text_to_Video_Generate_a_561_0_FLIPPED_pose.pt
✓  pred=2  true=2  |      1.6ms  |  2_dfsaeklnvvalkej_FLIPPED_pose.pt
✓  pred=4  true=4  |      1.6ms  |  4_kling_20251209_Text_to_Video_Generate_a_588_2_pose.pt
✓  pred=4  true=4  |      1.7ms  |  4_kling_20251209_Text_to_Video_Generate_a_452_1_FLIPPED_pose.pt
✓  pred=6  true=6  |      1.4ms  |  6_kling_20251209_Text_to_Video_Generate_a_218_1_pose.pt
✓  pred=3  true=3  |      1.5ms  |  3_sdlkjslndflkseijlkjef_FLIPPED_pose.pt
✓  pred=3  true=3  |      1.5ms  |  3_dkk873lkjlksajdf_FLIPPED_pose.pt
✓  pred=3  true=3  |      1.4ms  |  3_kling_20251206_Text_to_Video_Generate_a_315_0_FLIPPED_pose.pt
✓  pred=3  true=3  |      1.4ms  |  3_kling_20251209_Text_to_Video_Generate_a_491_2_pose.pt
✓  pred=4  true=4  |      1.5ms  |  4_aslkjasmcalkewjlkje_pose.pt
✓  pred=3  t